# Graph Algorithm Experiments — Territory Optimizer

Test graph-based approaches for territory optimisation:

### 1. Clustering algorithms
- **Baseline**: K-Means (current approach)
- **Spectral clustering** — graph Laplacian eigen decomposition
- **Agglomerative clustering** — hierarchical with connectivity constraints
- **DBSCAN** — density-based spatial clustering

### 2. Assignment algorithms
- **Baseline**: Greedy + local search (current)
- **Hungarian algorithm** — optimal linear assignment via scipy
- **Min-cost flow** — network flow on bipartite graph
- **MST assignment** — territory shapes from minimum spanning tree

### 3. Community detection & centrality
- **Label propagation** — semi-supervised graph diffusion
- **Bipartite spectral** — spectral clustering on dealer-projection graph
- **PageRank** — node centrality for ranking dealers/FTCs

All results are compared head-to-head with the same metrics.

---
## 0. Setup

In [ ]:
import subprocess, sys, importlib
for pkg in ['networkx']:
    try:
        importlib.import_module(pkg)
    except ImportError:
        print(f'Installing {pkg}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

import logging
logging.basicConfig(level=logging.WARNING, format='%(levelname)s %(message)s')
for lg in ['pipeline', 'optimization', 'features', 'data']:
    logging.getLogger(lg).setLevel(logging.WARNING)
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import networkx as nx
import time, math, json
from pathlib import Path
from copy import deepcopy
from itertools import combinations
from collections import deque

from sklearn.cluster import KMeans, SpectralClustering, AgglomerativeClustering, DBSCAN
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import silhouette_score
from sklearn.neighbors import kneighbors_graph
from sklearn.semi_supervised import LabelPropagation
from scipy.optimize import linear_sum_assignment
from scipy.sparse.csgraph import minimum_spanning_tree

%matplotlib inline
print('Setup OK')

In [ ]:
NUM_DEALERS = 300
NUM_FTCS = 15

data_dir = Path('data')
data_dir.mkdir(exist_ok=True)
for f in data_dir.glob('*.parquet'):
    f.unlink()

from data.generate_synthetic_data import SyntheticDataGenerator
gen = SyntheticDataGenerator(num_dealers=NUM_DEALERS, num_ftcs=NUM_FTCS, seed=42)
raw = gen.generate_all(output_dir='data')

from data.loader import DataLoader
from data.processor import DataProcessor
from config.settings import ConfigManager

config = ConfigManager()
loader = DataLoader()
data_raw = {
    'dealers': loader.load_dealers(),
    'ftcs': loader.load_ftcs(),
    'relationships': loader.load_relationships(),
    'proximity': loader.load_proximity(),
}
dealers = data_raw['dealers']
ftcs = data_raw['ftcs']
rels = data_raw['relationships']

proc = DataProcessor()
processed = proc.process_all_data(data_raw)
dealer_ids = processed['dealer_ids']
ftc_ids = processed['ftc_ids']

A = processed['assignment_matrix']  # [D x F] current assignment

print(f'Dealers:       {len(dealers)}  ({(dealers["dealer_type"]=="static").sum()} static)')
print(f'FTCs:          {len(ftcs)}')
print(f'Relationships: {len(rels)}')

In [ ]:
# Precompute all features for the experiments
from features.distance import DistanceEngineer
de = DistanceEngineer()
dist_km = de.compute_distance(dealers, ftcs, rels)  # [D x F] km

D = dist_km.shape[0]
F = dist_km.shape[1]

# Dealer-dealer distance matrix via vectorised Haversine
dealer_coords = dealers[['dealer_latitude', 'dealer_longitude']].values

def haversine_batch(lat1, lon1, lat2, lon2):
    R = 6371.0
    dlat = np.radians(lat2 - lat1)
    dlon = np.radians(lon2 - lon1)
    a = np.sin(dlat/2)**2 + np.cos(np.radians(lat1)) * np.cos(np.radians(lat2)) * np.sin(dlon/2)**2
    return R * 2 * np.arcsin(np.sqrt(a))

coords = dealer_coords
d_k = len(coords)
dd_km = np.zeros((d_k, d_k))
for i in range(d_k):
    dd_km[i] = haversine_batch(coords[i,0], coords[i,1], coords[:,0], coords[:,1])

print(f'Dealer-dealer matrix: {dd_km.shape}')

# Compatibility
from features.compatibility import CompatibilityEngineer
ce = CompatibilityEngineer()
compat = ce.compute_compatibility(dealers, ftcs, rels)  # [D x F]

# Workload + capacity
from features.workload import WorkloadEngineer
we = WorkloadEngineer()
workload = we.compute_workload(dealers)

from features.capacity import CapacityEngineer
cape = CapacityEngineer()
capacity = cape.compute_capacity(ftcs)

# Default objective weights
opt_cfg = config.get_section('optimization')
alpha_1 = opt_cfg.get('alpha_1', 0.5)
alpha_2 = opt_cfg.get('alpha_2', 0.3)
lambda_penalty = opt_cfg.get('lambda', 0.1)

print(f'Features ready: distance={dist_km.shape}, compat={compat.shape}')
print(f'Weights: alpha_1={alpha_1}, alpha_2={alpha_2}, lambda={lambda_penalty}')

In [ ]:
def evaluate_assignment(assign, label):
    num = len(assign)
    old = A.argmax(axis=1)
    changed = (assign != old).sum()
    dists = dist_km[np.arange(num), assign]
    compats = compat[np.arange(num), assign]
    active = len(set(assign))
    compat_norm = compats.mean()
    dist_norm = (dists / 100.0).mean()
    kept_ratio = 1.0 - changed / num
    objective = alpha_1 * compat_norm + alpha_2 * (1 - dist_norm) + lambda_penalty * kept_ratio
    return {
        'algorithm': label,
        'changed': int(changed),
        'change_rate': changed / num,
        'active_ftcs': active,
        'mean_dist_km': dists.mean(),
        'median_dist_km': np.median(dists),
        'mean_compat': compats.mean(),
        'objective': objective,
        'violations': int((dists > 100).sum()),
    }

def evaluate_clustering(labels, label, coords):
    n = len(set(labels))
    sizes = np.bincount(labels)
    sil = silhouette_score(coords, labels) if n > 1 and n < len(labels) else 0
    return {
        'algorithm': label,
        'num_clusters': n,
        'min_size': sizes.min(),
        'max_size': sizes.max(),
        'mean_size': sizes.mean(),
        'silhouette': sil,
    }

def print_assign(m):
    print(f'  {m["algorithm"]:30s}  changed={m["changed"]:4d}  '
          f'active={m["active_ftcs"]:2d}/{NUM_FTCS}  '
          f'dist={m["mean_dist_km"]:6.1f}km  '
          f'compat={m["mean_compat"]:.3f}  '
          f'obj={m["objective"]:.4f}  '
          f'viol={m["violations"]:3d}')

print('Eval helpers ready')

---
## 1. Graph Construction

Build the fundamental graphs used throughout the notebook.

In [ ]:
# Dealer proximity graph (threshold: 5 km)
PROX_THRESHOLD_KM = 5.0
G_dd = nx.Graph()
G_dd.add_nodes_from(range(D))
for i in range(D):
    for j in range(i + 1, D):
        if dd_km[i, j] <= PROX_THRESHOLD_KM and dd_km[i, j] > 0:
            G_dd.add_edge(i, j, weight=dd_km[i, j])
print(f'Proximity graph: {G_dd.number_of_nodes()} nodes, {G_dd.number_of_edges()} edges')

# K-nearest-neighbor graph (k=10)
K_NN = 10
G_knn = nx.Graph()
G_knn.add_nodes_from(range(D))
for i in range(D):
    nearest = np.argpartition(dd_km[i], K_NN + 1)[:K_NN + 1]
    for j in nearest:
        if j != i:
            G_knn.add_edge(i, j, weight=dd_km[i, j])
print(f'KNN graph:       {G_knn.number_of_nodes()} nodes, {G_knn.number_of_edges()} edges')

# Bipartite dealer-FTC graph
G_bf = nx.Graph()
for d in range(D):
    G_bf.add_node(f'd{d}', bipartite=0)
for f in range(F):
    G_bf.add_node(f'f{f}', bipartite=1)
for d in range(D):
    for f in range(F):
        w = alpha_1 * compat[d, f] + alpha_2 * (1 - dist_km[d, f] / dist_km.max())
        if w > 0:
            G_bf.add_edge(f'd{d}', f'f{f}', weight=w, dist_km=dist_km[d, f])
print(f'Bipartite graph:  {G_bf.number_of_nodes()} nodes, {G_bf.number_of_edges()} edges')

---
## 2. Graph-Based Clustering

Compare K-Means (current baseline) against spectral, agglomerative, and density-based clustering.

In [ ]:
N_CLUSTERS = max(1, min(int(NUM_FTCS * 1.5), D))
coords_scaled = StandardScaler().fit_transform(dealer_coords)
print(f'Target clusters: {N_CLUSTERS}')

# 2a. K-Means (baseline)
t0 = time.time()
km = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10)
km_labels = km.fit_predict(coords_scaled)
print(f'K-Means:         {time.time()-t0:.2f}s')

# 2b. Spectral clustering
t0 = time.time()
sc = SpectralClustering(n_clusters=N_CLUSTERS, affinity='nearest_neighbors',
                        n_neighbors=15, random_state=42, n_init=10)
sc_labels = sc.fit_predict(coords_scaled)
print(f'Spectral:        {time.time()-t0:.2f}s')

# 2c. Agglomerative clustering (hierarchical)
t0 = time.time()
ac = AgglomerativeClustering(n_clusters=N_CLUSTERS, linkage='ward')
ac_labels = ac.fit_predict(coords_scaled)
print(f'Agglomerative:   {time.time()-t0:.2f}s')

# 2d. Agglomerative with connectivity constraint (uses KNN graph)
t0 = time.time()
connectivity = kneighbors_graph(dealer_coords, n_neighbors=10, include_self=False)
ac_conn = AgglomerativeClustering(n_clusters=N_CLUSTERS, linkage='ward', connectivity=connectivity)
ac_conn_labels = ac_conn.fit_predict(coords_scaled)
print(f'Agglomerative+conn: {time.time()-t0:.2f}s')

# 2e. DBSCAN (density-based, auto-determines n_clusters)
t0 = time.time()
dealer_rad = np.radians(dealer_coords)
db = DBSCAN(eps=0.5, min_samples=5, metric='haversine')
db_labels = db.fit_predict(dealer_rad)
n_db_noise = list(db_labels).count(-1)
n_db = len(set(db_labels)) - (1 if -1 in db_labels else 0)
print(f'DBSCAN:          {time.time()-t0:.2f}s  ({n_db} clusters, {n_db_noise} noise)')

In [ ]:
clust_results = [
    evaluate_clustering(km_labels, 'K-Means (baseline)', coords_scaled),
    evaluate_clustering(sc_labels, 'Spectral', coords_scaled),
    evaluate_clustering(ac_labels, 'Agglomerative', coords_scaled),
    evaluate_clustering(ac_conn_labels, 'Agglomerative+conn', coords_scaled),
]
pd.DataFrame(clust_results)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
all_sets = [
    ('K-Means', km_labels),
    ('Spectral', sc_labels),
    ('Agglomerative', ac_labels),
    ('Agglomerative+conn', ac_conn_labels),
]
for ax, (title, labels) in zip(axes, all_sets):
    n = len(set(labels))
    cmap = plt.cm.tab20(np.linspace(0, 1, n))
    for i in range(D):
        ax.scatter(dealers.iloc[i]['dealer_longitude'], dealers.iloc[i]['dealer_latitude'],
                   c=[cmap[labels[i] % len(cmap)]], s=10, alpha=0.7)
    ax.set_title(f'{title} ({n} clusters)', fontsize=11, fontweight='bold')
    ax.set_xlabel('Lon'); ax.set_ylabel('Lat')
plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
core = db_labels != -1
noise = db_labels == -1
ax.scatter(dealers.iloc[core]['dealer_longitude'], dealers.iloc[core]['dealer_latitude'],
           c='steelblue', s=15, alpha=0.6, label=f'Core ({core.sum()})')
ax.scatter(dealers.iloc[noise]['dealer_longitude'], dealers.iloc[noise]['dealer_latitude'],
           c='red', s=30, alpha=0.8, marker='x', label=f'Noise ({noise.sum()})')
ax.set_title(f'DBSCAN (eps=0.5, min_samples=5) — {n_db} clusters')
ax.legend(); ax.set_xlabel('Lon'); ax.set_ylabel('Lat')
plt.tight_layout(); plt.show()

---
## 3. Graph-Based Assignment

Compare greedy solver against Hungarian, min-cost flow, and MST-based assignment.

In [ ]:
# 3a. Current assignment (baseline)
A_current = A.argmax(axis=1)
baseline = evaluate_assignment(A_current, 'Current (baseline)')
print_assign(baseline)

# 3b. Pipeline greedy solver
from pipeline import OptimizationPipeline
pipeline = OptimizationPipeline(config)
result = pipeline.run(parameters={'solver.time_limit_seconds': 30})
assign_greedy = A_current.copy()
sol_assign = result.get('assignments')
if sol_assign is not None:
    assign_greedy = np.array(sol_assign).argmax(axis=1)
greedy = evaluate_assignment(assign_greedy, 'Greedy (pipeline)')
print_assign(greedy)

In [ ]:
# 3c. Hungarian algorithm (optimal linear assignment)
t0 = time.time()
obj_score = alpha_1 * compat + alpha_2 * (1 - dist_km / dist_km.max())
cost_pad = np.full((max(D, F), max(D, F)), -obj_score.min() * 2)
cost_pad[:D, :F] = -obj_score  # Hungarian minimises
row_idx, col_idx = linear_sum_assignment(cost_pad)
hungarian_assign = col_idx[:D]
print(f'Hungarian:       {time.time()-t0:.2f}s')

ftc_counts = np.bincount(hungarian_assign, minlength=F)
overloaded = (ftc_counts > 25).sum()
hungarian = evaluate_assignment(hungarian_assign, 'Hungarian')
print_assign(hungarian)
if overloaded:
    print(f'  *** {overloaded} FTCs overloaded (>25) — Hungarian ignores capacity')

In [ ]:
# 3d. Min-cost flow (capacity-aware)
t0 = time.time()
G_flow = nx.DiGraph()
G_flow.add_node('source', demand=0)
G_flow.add_node('sink', demand=0)
for d in range(D):
    G_flow.add_node(f'd{d}', demand=0)
    G_flow.add_edge('source', f'd{d}', capacity=1, weight=0)
for f in range(F):
    G_flow.add_node(f'f{f}', demand=0)
    G_flow.add_edge(f'f{f}', 'sink', capacity=25, weight=0)
for d in range(D):
    feasible = np.argsort(dist_km[d])[:min(5, F)]  # top-5 nearest
    for f in feasible:
        G_flow.add_edge(f'd{d}', f'f{f}', capacity=1, weight=1.0 - obj_score[d, f])  # non-negative cost

try:
    flow_dict = nx.min_cost_flow(G_flow)
    mcf_assign = np.full(D, -1, dtype=int)
    for d in range(D):
        for f_str, flow in flow_dict[f'd{d}'].items():
            if flow > 0:
                mcf_assign[d] = int(f_str[1:])
    unassigned = (mcf_assign == -1).sum()
    mcf = evaluate_assignment(mcf_assign, 'Min-cost flow')
    print(f'Min-cost flow:   {time.time()-t0:.2f}s')
    print_assign(mcf)
    if unassigned:
        print(f'  *** {unassigned} dealers unassigned')
except Exception as e:
    print(f'Min-cost flow failed: {e}')
    mcf = None

In [ ]:
# 3e. MST-based territory assignment
t0 = time.time()
mst_matrix = minimum_spanning_tree(dd_km).toarray()
G_mst = nx.Graph()
for i in range(D):
    for j in range(i + 1, D):
        if mst_matrix[i, j] > 0:
            G_mst.add_edge(i, j, weight=mst_matrix[i, j])

mst_assign = np.full(D, -1, dtype=int)
ftc_seed = {}
for f in range(F):
    best_d = np.argmin(dist_km[:, f])
    if mst_assign[best_d] == -1:
        mst_assign[best_d] = f
        ftc_seed[f] = best_d

q = deque(ftc_seed.items())
while q:
    f, d = q.popleft()
    for neighbor in G_mst.neighbors(d):
        if mst_assign[neighbor] == -1:
            mst_assign[neighbor] = f
            q.append((f, neighbor))

for d in range(D):
    if mst_assign[d] == -1:
        mst_assign[d] = int(np.argmin(dist_km[d]))

print(f'MST assignment:  {time.time()-t0:.2f}s')
mst = evaluate_assignment(mst_assign, 'MST assignment')
print_assign(mst)

In [ ]:
assign_results = [baseline, greedy, hungarian, mst]
if mcf is not None:
    assign_results.append(mcf)
df_assign = pd.DataFrame(assign_results)

fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))
x = df_assign['algorithm']
metrics = [
    ('mean_dist_km', 'Mean Distance (km)', 'steelblue'),
    ('mean_compat', 'Mean Compatibility', 'coral'),
    ('objective', 'Objective Score', 'mediumseagreen'),
    ('changed', 'Dealers Reassigned', 'mediumpurple'),
]
for ax, (col, ylabel, color) in zip(axes, metrics):
    bars = ax.bar(x, df_assign[col], color=color, edgecolor='black', linewidth=0.5)
    ax.set_ylabel(ylabel)
    ax.set_title(ylabel, fontsize=11, fontweight='bold')
    ax.tick_params(axis='x', rotation=20)
    for bar, v in zip(bars, df_assign[col]):
        val = f'{v:.1f}' if isinstance(v, float) else str(v)
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()*1.01, val,
                ha='center', va='bottom', fontsize=9)
plt.suptitle('Assignment Algorithm Comparison', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

df_assign

---
## 4. Community Detection

Use graph-based semi-supervised learning and spectral methods on the bipartite projection.

In [ ]:
# 4a. Label propagation on the proximity affinity matrix
t0 = time.time()
W = np.exp(-dd_km / 10.0)  # affinity from distance
np.fill_diagonal(W, 1)

y_seed = np.full(D, -1, dtype=int)
for f in range(F):
    best_d = np.argmin(dist_km[:, f])
    y_seed[best_d] = f

lp = LabelPropagation(kernel='knn', n_neighbors=10, max_iter=100)
lp.fit(W, y_seed)
lp_labels = lp.transduction_
print(f'Label propagation: {time.time()-t0:.2f}s  ({len(set(lp_labels))} groups)')
lp_result = evaluate_assignment(lp_labels, 'Label propagation')
print_assign(lp_result)

In [ ]:
# 4b. Spectral clustering on bipartite projection graph
t0 = time.time()
G_proj = nx.Graph()
G_proj.add_nodes_from(range(D))
for f in range(F):
    dealers_at_ftc = A[:, f].nonzero()[0]
    for i, j in combinations(dealers_at_ftc, 2):
        if G_proj.has_edge(i, j):
            G_proj[i][j]['weight'] += 1
        else:
            G_proj.add_edge(i, j, weight=1)

adj = nx.to_numpy_array(G_proj, nodelist=range(D), weight='weight')
sc_proj = SpectralClustering(n_clusters=F, affinity='precomputed', random_state=42)
proj_labels = sc_proj.fit_predict(adj)
print(f'Bipartite spectral: {time.time()-t0:.2f}s')
proj_result = evaluate_assignment(proj_labels, 'Bipartite spectral')
print_assign(proj_result)

---
## 5. Graph Centrality Analysis

Use graph centrality metrics to identify key nodes — which dealers or FTCs are most influential.

In [ ]:
# Degree centrality on the bipartite graph
bf_deg = dict(G_bf.degree())
dealer_centrality = np.array([bf_deg.get(f'd{d}', 0) for d in range(D)])
ftc_centrality = np.array([bf_deg.get(f'f{f}', 0) for f in range(F)])

print('Top 5 dealers by degree:')
for d in np.argsort(-dealer_centrality)[:5]:
    print(f'  {dealer_ids[d]:12s}  degree={dealer_centrality[d]}  '
          f'cases={dealers.iloc[d]["average_cases_per_day"]:.1f}')
print()
print('FTCs by degree:')
for f in np.argsort(-ftc_centrality):
    print(f'  {ftc_ids[f]:12s}  degree={ftc_centrality[f]}')

In [ ]:
# Closeness centrality on the KNN graph
if G_knn.number_of_nodes() <= 1000:
    t0 = time.time()
    close = nx.closeness_centrality(G_knn)
    print(f'Closeness centrality: {time.time()-t0:.2f}s')

    fig, ax = plt.subplots(figsize=(10, 6))
    vals = np.array([close.get(i, 0) for i in range(D)])
    sc = ax.scatter(dealers['dealer_longitude'], dealers['dealer_latitude'],
                    c=vals, cmap='viridis', s=25, alpha=0.7, edgecolors='black', linewidths=0.2)
    plt.colorbar(sc, ax=ax, label='Closeness centrality')
    ax.set_title('Closeness Centrality — KNN Graph', fontsize=12, fontweight='bold')
    ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
    plt.tight_layout(); plt.show()

    print('Most central dealers:')
    for i, v in sorted(close.items(), key=lambda x: -x[1])[:10]:
        print(f'  {dealer_ids[i]:12s}  closeness={v:.4f}')

---
## 6. Head-to-Head Comparison

All graph-based algorithms ranked by objective score.

In [ ]:
all_comparisons = [baseline, greedy, hungarian, mst, lp_result, proj_result]
if mcf is not None:
    all_comparisons.append(mcf)

df_comp = pd.DataFrame(all_comparisons).sort_values('objective', ascending=False).reset_index(drop=True)
df_comp.index = df_comp.index + 1
df_comp.index.name = 'Rank'
df_comp

In [ ]:
print('=' * 72)
print(f'BEST objective:   {df_comp.iloc[0]["algorithm"]}  ({df_comp.iloc[0]["objective"]:.4f})')
print(f'BEST distance:    {df_comp.loc[df_comp["mean_dist_km"].idxmin(), "algorithm"]}  '
      f'({df_comp["mean_dist_km"].min():.1f}km)')
print(f'BEST compat:      {df_comp.loc[df_comp["mean_compat"].idxmax(), "algorithm"]}  '
      f'({df_comp["mean_compat"].max():.3f})')
print(f'BEST low change:  {df_comp.loc[df_comp["changed"].idxmin(), "algorithm"]}  '
      f'({df_comp["changed"].min()} dealers)')
print(f'BEST active FTCs: {df_comp.loc[df_comp["active_ftcs"].idxmax(), "algorithm"]}  '
      f'({df_comp["active_ftcs"].max()})')
print('=' * 72)

In [ ]:
# Map the best result
best_alg = df_comp.iloc[0]['algorithm']
best_map = {
    'Hungarian': hungarian_assign,
    'Min-cost flow': mcf_assign if mcf is not None else None,
    'MST assignment': mst_assign,
    'Label propagation': lp_labels,
    'Bipartite spectral': proj_labels,
    'Greedy (pipeline)': assign_greedy,
}
best_assign = best_map.get(best_alg, assign_greedy)

palette = plt.cm.Set1(np.linspace(0, 1, F))
ftc_color = dict(zip(range(F), palette))
old_assign = A.argmax(axis=1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))
for title, ax, assign in [('Before (Current)', ax1, old_assign), (f'After ({best_alg})', ax2, best_assign)]:
    for i in range(D):
        c = ftc_color[assign[i] % F]
        ax.scatter(dealers.iloc[i]['dealer_longitude'], dealers.iloc[i]['dealer_latitude'],
                   c=[c], s=15, alpha=0.7, edgecolors='black', linewidths=0.2)
    for f in range(F):
        lat, lon = ftcs.iloc[f]['ftc_latitude'], ftcs.iloc[f]['ftc_longitude']
        ax.scatter(lon, lat, c='green', s=120, marker='^', edgecolors='black', linewidths=0.5, zorder=5)
        ax.annotate(ftc_ids[f], (lon, lat),
                    textcoords='offset points', xytext=(0, 10), ha='center', fontsize=7, fontweight='bold',
                    bbox=dict(boxstyle='round,pad=0.15', facecolor='white', edgecolor='gray', alpha=0.7))
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')

handles = [mpatches.Patch(color=ftc_color[f], label=ftc_ids[f], linewidth=0.5, edgecolor='black') for f in range(F)]
handles.append(plt.Line2D([0], [0], marker='^', color='w', markerfacecolor='green', markersize=10, label='FTC'))
fig.legend(handles=handles, loc='lower center', ncol=min(F + 1, 10), fontsize=7, framealpha=0.9)
plt.tight_layout(rect=[0, 0.06, 1, 1])
plt.show()

---
## 7. Custom Experiment

Try your own graph-based approach. Available data: `dist_km`, `compat`, `dd_km`, `G_dd`, `G_knn`, `G_bf`.

In [ ]:
# Example: PageRank-weighted assignment
t0 = time.time()
pr = nx.pagerank(G_bf, alpha=0.85, max_iter=100)
custom_assign = np.array([
    np.argmax([pr.get(f'f{f}', 0) * compat[d, f] for f in range(F)])
    for d in range(D)
])
custom = evaluate_assignment(custom_assign, 'PageRank * compat')
print(f'Custom:          {time.time()-t0:.2f}s')
print_assign(custom)

In [ ]:
all_comparisons.append(custom)
df_final = pd.DataFrame(all_comparisons).sort_values('objective', ascending=False).reset_index(drop=True)
df_final.index = df_final.index + 1
df_final.index.name = 'Rank'
df_final

---
## 8. Algorithm Reference

| Algorithm | Type | Pros | Cons |
|-----------|------|------|------|
| **K-Means** | Clustering | Fast, scalable, well-understood | Spherical clusters only, needs n_clusters |
| **Spectral** | Clustering | Handles non-convex shapes, graph-based | Expensive for large n, needs affinity matrix |
| **Agglomerative** | Clustering | Hierarchical, dendrogram interpretable | O(n³), hard to choose linkage |
| **DBSCAN** | Clustering | No n_clusters needed, finds outliers | Struggles with varying density |
| **Hungarian** | Assignment | Globally optimal per-dealer assignment | No capacity constraints, O(n³) |
| **Min-cost flow** | Assignment | Respects capacity, network optimal | Needs feasible flow, can leave dealers unassigned |
| **Greedy+LS** | Assignment | Fast, capacity-aware, handles clusters | Local optima, heuristic |
| **MST** | Assignment | Topology-aware, natural territories | Sensitive to seed selection |
| **Label prop.** | Community | Fast, no n_clusters needed | Sensitive to seeds, may collapse |
| **PageRank** | Centrality | Identifies influential nodes | Not directly an assignment method |

**Graphs built**:
- `G_dd`: dealer-proximity (distance ≤ 5 km)
- `G_knn`: 10-nearest-neighbor graph
- `G_bf`: bipartite dealer-to-FTC graph (edge weight = objective score)
- `G_proj`: dealer projection from shared FTCs
- `G_mst`: minimum spanning tree of dealer distances